# Assessment of FAIR Metadata scores for CoMSES.net,  CSDMS and HydroShare model repositories

This notebook is intended to probe the FAIR evaluation results for three model repositories (CoMSES.net, CSDMS and HydroShare) by programmatically finding an example "low" and "high" FAIR-ness model, manually assessing the metadata in each and communicate observations. 

## Import libraries

In [1]:
import os
import json
import glob
import pandas as pd
import matplotlib.pyplot as plt

## Retrieve evaluator results and calculate overall FAIR score

In [3]:
# for storing overall results
res = []

# iterate across repositories
repositories = ['comses', 'csdms', 'hydroshare']

for repository in repositories:
    # get path for each model
    folder_name = 'out_' + repository
    path = os.path.join(folder_name, '*.json')
    # iterate across models in repository
    for filepath in glob.glob(path):
        filename = os.path.basename(filepath)
        model_id = os.path.splitext(filename)[0]
        
        with open(filepath, 'r') as f:
            # unpack score from JSON
            data = json.load(f)
            model_name = [*data][0]
            scores = data[model_name]['scores']
            
            f_score = scores['F']
            a_score = scores['A']
            i_score = scores['I']
            r_score = scores['R']

            # overall score is the sum of F, A, I and R
            # theorectical maximum is 4
            overall_score = f_score + a_score + i_score + r_score
            
            res.append({
                'repository': repository,
                'model_id': model_id,
                'overall_score': overall_score
            })

In [4]:
res_df = pd.DataFrame.from_dict(res)
print(res_df.head())

  repository                              model_id  overall_score
0     comses  003518b1-a9e5-4fe8-b76d-97d81d0be6b6           1.50
1     comses  0065a103-490a-4e5e-a234-7a4376fc33c2           1.50
2     comses  00ef291a-3dfc-401a-b80a-99062cd93858           1.00
3     comses  01a3a4f9-26ec-4a93-9d32-9e438a54cc7b           1.58
4     comses  02594069-f5bb-42ba-9c47-c907a6e264b8           2.08


## Select lowest and highest performing models in each repository

In [10]:
for repository in repositories:
    
    subset = res_df[res_df['repository'] == repository]
    
    min_score = subset['overall_score'].min()
    low_scoring_model = subset[subset['overall_score'] == min_score].iloc[0]

    max_score = subset['overall_score'].max()
    high_scoring_model = subset[subset['overall_score'] == max_score].iloc[0]
    
    print(f"Minimum score for {repository}: {min_score} (e.g., {low_scoring_model['model_id']})")
    print(f"Minimum score for {repository}: {max_score} (e.g., {high_scoring_model['model_id']})\n")

    if repository == 'hydroshare':
        filepath = os.path.join('out_hydroshare', low_scoring_model['model_id'] + '.json')
        with open(filepath, 'r') as f:
            # unpack score from JSON
            log = json.load(f)
            print(log)
            print()

Minimum score for comses: 0.875 (e.g., 5903)
Minimum score for comses: 2.58 (e.g., 12e0b16d-768b-4624-8af0-4c2dbe52d4e5)

Minimum score for csdms: 1.125 (e.g., ACADIA)
Minimum score for csdms: 2.795 (e.g., TIN-based Real-time Integrated Basin Simulator (tRIBS))

Minimum score for hydroshare: 1.375 (e.g., a6eb75fcb1684cf29da143f87c4be28c-model-program)
Minimum score for hydroshare: 2.42 (e.g., 0e32189bd6f84d2d8b2f76cce06e3beb-model-program)

{'model-program': {'indicator_evaluations': {'A1_1': {'result': True, 'log': ["http://creativecommons.org/licenses/by-nc-sa/4.0/ in codemeta property 'license' is an open link to the software.", "https://www.hydroshare.org/resource/a6eb75fcb1684cf29da143f87c4be28c in codemeta property 'url' is an open link to the software."]}, 'A1_2': {'result': False, 'log': ['This resource is shared under the Creative Commons Attribution-NoCommercial-ShareAlike CC BY-NC-SA. is NOT an approved license.']}, 'A2': {'result': True, 'log': ["https://www.hydroshare.org/

In [23]:
import pandas as pd
import json
import os
from IPython.display import display

# Mapping of indicators to condensed summaries derived from evaluators.py
# This ensures each indicator has the context of the FAIR sub-principle it tests
INDICATOR_INFO = {
    'F1_1': 'Unique/Distinct identifiers for software components',
    'F1_2': 'Semantic versioning and distinct version identifiers',
    'F2': 'Rich metadata descriptions (e.g., Codemeta/Codemeticulous)',
    'F3': 'Metadata includes the unique persistent identifier (PID)',
    'F4': 'Metadata is registered or indexed in a FAIR-aligned repository',
    'A1_1': 'Open and free communication protocols for software access',
    'A1_2': 'Explicitly described conditions for access (licensing/paywalls)',
    'A2': 'Metadata persistence even if software is no longer available',
    'I1': 'Use of standard data formats and documented APIs',
    'I2': 'Qualified references to external data/digital objects',
    'R1_1': 'Use of unrestrictive, community-recognized licenses',
    'R1_2': 'Clear provenance (contributors, history, and lineage)',
    'R2': 'Qualified references to software dependencies',
    'R3': 'Compliance with domain-relevant community standards'
}

# The definitive order for the FAIR principles
FAIR_ORDER = ['F1_1', 'F1_2', 'F2', 'F3', 'F4', 'A1_1', 'A1_2', 'A2', 'I1', 'I2', 'R1_1', 'R1_2', 'R2', 'R3']

def extract_log_data(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)
        model_name = [*data][0]
        evaluations = data[model_name]['indicator_evaluations']
        
        rows = []
        for indicator, details in evaluations.items():
            log_text = " | ".join(details['log']) 
            rows.append({
                'Indicator': indicator,
                'Description': INDICATOR_INFO.get(indicator, 'N/A'),
                'Result': 'Pass' if details['result'] else 'Fail',
                'Log': log_text
            })
        return rows

for repository in repositories:
    subset = res_df[res_df['repository'] == repository]
    
    min_score = subset['overall_score'].min()
    low_scoring_model = subset[subset['overall_score'] == min_score].iloc[0]

    max_score = subset['overall_score'].max()
    high_scoring_model = subset[subset['overall_score'] == max_score].iloc[0]
    
    print(f"--- COMPARISON ANALYSIS: {repository.upper()} ---")
    print(f"HIGH PERFORMER: {high_scoring_model['model_id']} | LOW PERFORMER: {low_scoring_model['model_id']}\n")

    if repository == 'hydroshare':
        low_filepath = os.path.join('out_hydroshare', low_scoring_model['model_id'] + '.json')
        high_filepath = os.path.join('out_hydroshare', high_scoring_model['model_id'] + '.json')
        
        # 1. Extract raw data
        high_data = extract_log_data(high_filepath)
        low_data = extract_log_data(low_filepath)
        
        # 2. Convert to DataFrames and merge
        df_high = pd.DataFrame(high_data).rename(columns={'Result': 'Result (High)', 'Log': 'Log (High)'})
        df_low = pd.DataFrame(low_data).rename(columns={'Result': 'Result (Low)', 'Log': 'Log (Low)'})
        df_merged = pd.merge(df_high, df_low, on=['Indicator', 'Description'])
        
        # 3. ENFORCE THE FAIR ORDER
        df_merged['Indicator'] = pd.Categorical(df_merged['Indicator'], categories=FAIR_ORDER, ordered=True)
        df_merged = df_merged.sort_values('Indicator')
        
        # 4. Styling for Heatmap effect
        def color_result(val):
            if val == 'Pass':
                return 'background-color: #d4edda; color: #155724; font-weight: bold; border: 1px solid #c3e6cb;'
            if val == 'Fail':
                return 'background-color: #f8d7da; color: #721c24; font-weight: bold; border: 1px solid #f5c6cb;'
            return ''
        
        # 5. Render final styled table
        styled_df = df_merged.style.applymap(color_result, subset=['Result (High)', 'Result (Low)']) \
            .set_properties(subset=['Indicator'], **{'font-weight': 'bold', 'text-align': 'center'}) \
            .set_properties(subset=['Description'], **{'width': '180px', 'text-align': 'left', 'font-size': '11px'}) \
            .set_properties(subset=['Log (High)', 'Log (Low)'], **{'width': '280px', 'text-align': 'left', 'white-space': 'pre-wrap', 'font-size': '10.5px'}) \
            .set_table_styles([
                {'selector': 'th', 'props': [('background-color', '#444'), ('color', 'white'), ('text-align', 'center')]},
                {'selector': 'td', 'props': [('vertical-align', 'top')]}
            ]) \
            .hide()
        
        display(styled_df)

--- COMPARISON ANALYSIS: COMSES ---
HIGH PERFORMER: 12e0b16d-768b-4624-8af0-4c2dbe52d4e5 | LOW PERFORMER: 5903

--- COMPARISON ANALYSIS: CSDMS ---
HIGH PERFORMER: TIN-based Real-time Integrated Basin Simulator (tRIBS) | LOW PERFORMER: ACADIA

--- COMPARISON ANALYSIS: HYDROSHARE ---
HIGH PERFORMER: 0e32189bd6f84d2d8b2f76cce06e3beb-model-program | LOW PERFORMER: a6eb75fcb1684cf29da143f87c4be28c-model-program



C:\Users\AbnerBogan\AppData\Local\Temp\ipykernel_32252\614553800.py:83: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled_df = df_merged.style.applymap(color_result, subset=['Result (High)', 'Result (Low)']) \


Indicator,Description,Result (High),Log (High),Result (Low),Log (Low)
F1_1,Unique/Distinct identifiers for software components,Pass,https://doi.org/10.4211/hs.0e32189bd6f84d2d8b2f76cce06e3beb in codemeta property 'identifier' is a distinct identifier.,Pass,https://hydroshare.org/resource/a6eb75fcb1684cf29da143f87c4be28c in codemeta property 'identifier' is a distinct identifier. | https://www.hydroshare.org/resource/a6eb75fcb1684cf29da143f87c4be28c in codemeta property 'url' is a distinct identifier.
F1_2,Semantic versioning and distinct version identifiers,Pass,2.0.0 in codemeta property 'softwareVersion' represents valid semantic versioning.,Fail,1.0 in codemeta property 'softwareVersion' does NOT represents valid semantic versioning.
F2,"Rich metadata descriptions (e.g., Codemeta/Codemeticulous)",Fail,"Missing inputs ['description', 'fileSize'] for evalF2",Fail,"Missing inputs ['description', 'fileSize'] for evalF2"
F3,Metadata includes the unique persistent identifier (PID),Pass,https://doi.org/10.4211/hs.0e32189bd6f84d2d8b2f76cce06e3beb in codemeta property 'identifier' is a globally unique and persistent identifier for software.,Pass,https://hydroshare.org/resource/a6eb75fcb1684cf29da143f87c4be28c in codemeta property 'identifier' is a globally unique and persistent identifier for software.
F4,Metadata is registered or indexed in a FAIR-aligned repository,Pass,https://doi.org/10.4211/hs.0e32189bd6f84d2d8b2f76cce06e3beb does point to software metadata in FAIR-aligned repository.,Pass,https://hydroshare.org/resource/a6eb75fcb1684cf29da143f87c4be28c does point to software metadata in FAIR-aligned repository.
A1_1,Open and free communication protocols for software access,Pass,http://creativecommons.org/licenses/by/4.0/ in codemeta property 'license' is an open link to the software. | http://www.ral.ucar.edu/projects/summa in codemeta property 'url' is an open link to the software.,Pass,http://creativecommons.org/licenses/by-nc-sa/4.0/ in codemeta property 'license' is an open link to the software. | https://www.hydroshare.org/resource/a6eb75fcb1684cf29da143f87c4be28c in codemeta property 'url' is an open link to the software.
A1_2,Explicitly described conditions for access (licensing/paywalls),Pass,This resource is shared under the Creative Commons Attribution CC BY. is an approved license.,Fail,This resource is shared under the Creative Commons Attribution-NoCommercial-ShareAlike CC BY-NC-SA. is NOT an approved license.
A2,Metadata persistence even if software is no longer available,Pass,https://doi.org/10.4211/hs.0e32189bd6f84d2d8b2f76cce06e3beb in codemeta property 'identifier' is providing access to the software metadata in an approved software reigstry.,Pass,https://www.hydroshare.org/resource/a6eb75fcb1684cf29da143f87c4be28c in codemeta property 'url' is providing access to the software metadata in an approved software reigstry.
I1,Use of standard data formats and documented APIs,Fail,"Missing inputs ['supportingData', 'buildInstructions'] for evalI1",Fail,"Missing inputs ['supportingData', 'buildInstructions'] for evalI1"
I2,Qualified references to external data/digital objects,Fail,"Missing inputs ['relatedLink', 'supportingData', 'isPartOf', 'referencePublication'] for evalI2",Fail,"Missing inputs ['relatedLink', 'supportingData', 'isPartOf', 'referencePublication'] for evalI2"


## Document observations for all models